# GNN fraud detection on Elliptic — Colab runner

Runs the whole pipeline on a free T4: download the data, train GraphSAGE, run the GAN augmentation, export to ONNX and benchmark the Lambda handler.

Set the runtime to GPU first: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 1. Install dependencies

In [ ]:
!pip install -q torch-geometric onnx onnxruntime onnxscript kaggle

## 2. Clone the repo

In [ ]:
# replace with your fork's URL
!git clone https://github.com/<your-username>/gnn-fraud-detection.git
%cd gnn-fraud-detection
!mkdir -p data/elliptic_real models outputs

## 3. Download the Elliptic dataset

Grab an API token from https://www.kaggle.com/settings → API → Create New Token, then paste it when prompted.

In [ ]:
import os, getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Kaggle API token: ')

In [ ]:
!kaggle datasets download -d ellipticco/elliptic-data-set -p /tmp/elliptic --unzip -q

import glob, shutil
for name in ['elliptic_txs_features.csv', 'elliptic_txs_classes.csv', 'elliptic_txs_edgelist.csv']:
    src = glob.glob(f'/tmp/elliptic/**/{name}', recursive=True)[0]
    shutil.copy(src, 'data/elliptic_real/')
!ls -lh data/elliptic_real

## 4. Train the GraphSAGE baseline

In [ ]:
%cd src
!python train_baseline.py --data_dir ../data/elliptic_real --epochs 150

## 5. GAN augmentation

In [ ]:
!python train_gan_augmented.py --data_dir ../data/elliptic_real --gan_epochs 500 --head_epochs 150

In [ ]:
import json
r = json.load(open('../outputs/gan_augmented_metrics.json'))
b, g = r['baseline_test_metrics'], r['gan_augmented_test_metrics']
print(f"{'metric':<22}{'baseline':>10}{'+ GAN':>10}")
for k in ['f1_illicit', 'precision_illicit', 'recall_illicit', 'false_positive_rate', 'roc_auc']:
    print(f"{k:<22}{b[k]:>10.4f}{g[k]:>10.4f}")
print(f"\nFPR reduction: {r['false_positive_rate_change_pct']:.1f}%")

## 6. Export to ONNX and benchmark the Lambda handler

In [ ]:
%cd ../lambda
!python export_to_onnx.py --ckpt ../models/baseline_sage.pt --out model.onnx

In [ ]:
import os, json, time, importlib
import numpy as np
import onnxruntime as ort

os.environ['MODEL_PATH'] = 'model.onnx'
import lambda_handler
importlib.reload(lambda_handler)

n_feat = ort.InferenceSession('model.onnx').get_inputs()[0].shape[1]
rng = np.random.default_rng(0)
n = 100
event = {
    'node_features': rng.standard_normal((n, n_feat)).tolist(),
    'edge_index': rng.integers(0, n, (2, 300)).tolist(),
}

for _ in range(5):  # warm up
    lambda_handler.handler(event, None)

times = []
for _ in range(100):
    t0 = time.perf_counter()
    resp = lambda_handler.handler(event, None)
    times.append((time.perf_counter() - t0) * 1000)

print(json.loads(resp['body']))
print(f'{n}-node subgraph: mean {np.mean(times):.2f} ms | p50 {np.median(times):.2f} ms | p99 {np.percentile(times, 99):.2f} ms')

## 7. Download outputs

In [ ]:
from google.colab import files
%cd ..
for p in ['outputs/baseline_metrics.json', 'outputs/gan_augmented_metrics.json', 'lambda/model.onnx']:
    if os.path.exists(p):
        files.download(p)